# Model

### Imports

In [17]:
import sys
import pandas as pd
import xgboost as xgb
import optuna
import numpy as np

from optuna.integration import XGBoostPruningCallback

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    matthews_corrcoef
)
from sklearn.ensemble import (
    RandomForestClassifier,
    BaggingClassifier,
    VotingClassifier
)

### Load Dataset

In [18]:
# Allow the dataset to be loaded with both a Google Colab kernel and a local kernel

# Load the dataset with Google Colab kernel and Drive file
if 'google.colab' in sys.modules:
    # Time (aprox. 40.0s)
    from pydrive2.auth import GoogleAuth
    from pydrive2.drive import GoogleDrive
    from google.colab import auth # type: ignore
    from oauth2client.client import GoogleCredentials

    # Authenticate the User in Google Drive
    auth.authenticate_user()
    gauth = GoogleAuth()
    gauth.credentials = GoogleCredentials.get_application_default()
    drive = GoogleDrive(gauth)

    # Google Drive ID for public sharing of the dataset

    # processed_data.csv
    file_id = "1jNCvMvAPawvH8avENiwbYvQReaZOiqMH"
    file = drive.CreateFile({'id': file_id})
    file.GetContentFile('processed_data.csv')

    # sampled_data.csv
    file_id = "1h03hkE5jMsG_17yFSUeNXNnc-RxzZxQJ"
    file = drive.CreateFile({'id': file_id})
    file.GetContentFile('sampled_data.csv')

    # Reading the csv and loading it into a pandas dataframe (Use pyarrow to prevent OOM error when loading)
    df = pd.read_csv("processed_data.csv", engine="pyarrow", dtype_backend="pyarrow")
    df_sampled = pd.read_csv("sampled_data.csv", engine="pyarrow", dtype_backend="pyarrow")

# Load the dataset with local kernel and local file
else:
    # Time (13th Gen Intel Core i5-1335U: aprox. 1.0s; AMD Ryzen AI 9 HX 370 (24) @ 5.16 GHz: aprox. 0.4s)
    # Reading the csv and loading it into a pandas dataframe (Use pyarrow to prevent OOM error when loading)
    df = pd.read_csv("../data/processed/processed_data.csv", engine="pyarrow", dtype_backend="pyarrow")
    df_sampled = pd.read_csv("../data/sampled/sampled_data.csv", engine="pyarrow", dtype_backend="pyarrow")

# Stop Jupyter Notebook from limiting the output
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Display dataset head
df.head()
df_sampled.head()

,loan_amnt,term,int_rate,installment,sub_grade,emp_length,home_ownership,annual_inc,purpose,addr_state,dti,delinq_2yrs,fico_range_low,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,mths_since_last_major_derog,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_act_il,il_util,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,acc_open_past_24mths,bc_open_to_buy,bc_util,chargeoff_within_12_mths,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,issue_d_year,issue_d_month,earliest_cr_line_year,earliest_cr_line_month,loan_status
0,7200.0,0.0,9.16,229.5,6.0,2.0,1.0,63500.0,2.0,23.0,32.32,2.0,675.0,2.0,15.0,999.0,15.0,0.0,31543.0,90.6,23.0,0.0,18.0,0.0,0.0,0.0,96458.0,9.0,93.0,0.0,25225.0,92.0,34800.0,0.0,0.0,3.0,804.0,97.5,0.0,152.0,228.0,27.0,16.0,0.0,27.0,15.0,5.0,15.0,1.0,5.0,6.0,5.0,7.0,13.0,6.0,9.0,6.0,15.0,91.3,80.0,0.0,0.0,104840.0,96458.0,32100.0,70040.0,2016.0,2.0,2016.0,2.0,0
1,30000.0,0.0,14.64,1034.68,12.0,10.0,2.0,95000.0,2.0,38.0,7.04,1.0,665.0,0.0,2.0,999.0,9.0,0.0,5287.0,41.0,17.0,1.0,999.0,0.0,0.0,0.0,22354.0,0.0,74.0,2.0,4187.0,60.0,12900.0,0.0,0.0,1.0,1819.0,74.0,0.0,112.0,322.0,72.0,3.0,3.0,72.0,2.0,16.0,2.0,1.0,3.0,4.0,3.0,6.0,4.0,6.0,9.0,4.0,9.0,88.2,66.7,0.0,0.0,40927.0,22354.0,7000.0,18510.0,2014.0,2.0,2014.0,3.0,0
2,10000.0,0.0,16.99,356.48,17.0,10.0,1.0,48000.0,2.0,43.0,33.2,0.0,670.0,0.0,50.0,999.0,13.0,0.0,7881.0,28.3,24.0,1.0,50.0,0.0,0.0,7738.0,41489.0,0.0,74.0,2.0,4187.0,60.0,27800.0,0.0,0.0,9.0,17919.0,30.5,0.0,43.0,135.0,10.0,2.0,0.0,10.0,71.0,10.0,50.0,5.0,2.0,2.0,9.0,15.0,3.0,11.0,19.0,2.0,13.0,79.2,11.1,0.0,0.0,62800.0,41489.0,25800.0,35000.0,2015.0,9.0,2015.0,5.0,0
3,13000.0,1.0,7.49,260.44,3.0,10.0,2.0,65000.0,7.0,34.0,3.43,0.0,780.0,0.0,999.0,999.0,14.0,0.0,31658.0,37.9,26.0,1.0,999.0,0.0,0.0,0.0,79772.0,0.0,74.0,2.0,4187.0,60.0,24100.0,0.0,0.0,0.0,4702.0,63.1,0.0,129.0,164.0,8.0,5.0,0.0,13.0,999.0,999.0,999.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,97.9,42.9,0.0,0.0,111963.0,37338.0,15100.0,31759.0,2011.0,9.0,2011.0,12.0,0
4,5000.0,0.0,10.91,163.49,8.0,8.0,2.0,75000.0,4.0,47.0,5.1,0.0,670.0,0.0,999.0,999.0,5.0,0.0,376.0,2.0,9.0,0.0,999.0,0.0,0.0,189.0,217482.0,1.0,67.0,3.0,333.0,30.0,19000.0,0.0,0.0,5.0,18624.0,2.0,0.0,137.0,53.0,1.0,1.0,1.0,1.0,999.0,7.0,999.0,0.0,2.0,2.0,3.0,5.0,3.0,3.0,5.0,2.0,5.0,100.0,0.0,0.0,0.0,242244.0,10014.0,19000.0,14418.0,2018.0,2.0,2018.0,9.0,0


### Load Indexes

In [19]:
# Load the indexes with Google Colab kernel and Drive file
if 'google.colab' in sys.modules:
    # Time (aprox. 5.0s)
    from pydrive2.auth import GoogleAuth
    from pydrive2.drive import GoogleDrive
    from google.colab import auth # type: ignore
    from oauth2client.client import GoogleCredentials

    # Authenticate the User in Google Drive
    auth.authenticate_user()
    gauth = GoogleAuth()
    gauth.credentials = GoogleCredentials.get_application_default()
    drive = GoogleDrive(gauth)

    # Google Drive ID for public sharing of the indexes

    # train_indexes.txt
    file_id = "18QUVIm8wtVeGQNJ0oaiXz-X1rqduqeOk"
    file = drive.CreateFile({'id': file_id})
    file.GetContentFile('train_indexes.txt')

    with open('train_indexes.txt', 'r') as f:
        train_indexes = [int(line.strip()) for line in f]

    # eval_indexes.txt
    file_id = "1WShrO8AJ57AUf7LxbZCw7pNc-pd5ZQEo"
    file = drive.CreateFile({'id': file_id})
    file.GetContentFile('eval_indexes.txt')

    with open("eval_indexes.txt", "r") as f:
        eval_indexes = [int(line.strip()) for line in f]

    # test_indexes.txt
    file_id = "1dL9bkv-wEISbFau4tAzkzEgZRHbnwc7t"
    file = drive.CreateFile({'id': file_id})
    file.GetContentFile('test_indexes.txt')

    with open("test_indexes.txt", "r") as f:
        test_indexes = [int(line.strip()) for line in f]

# Load the indexes with local kernel and local file
else:
    # Time (13th Gen Intel Core i5-1335U: aprox. 0.1s; AMD Ryzen AI 9 HX 370 (24) @ 5.16 GHz: aprox. 0.0s)
    with open("../data/indexes/train_indexes.txt", "r") as f:
        train_indexes = [int(line.strip()) for line in f]
    with open("../data/indexes/eval_indexes.txt", "r") as f:
        eval_indexes = [int(line.strip()) for line in f]
    with open("../data/indexes/test_indexes.txt", "r") as f:
        test_indexes = [int(line.strip()) for line in f]

### Model Training

Dataset Split and Define Metrics:

In [20]:
# Split the Dataset into X and y (loan_status)
# processed_data
X = df.drop(columns=["loan_status"])
y = df["loan_status"]

# sampled_data
X_sampled = df_sampled.drop(columns=["loan_status"])
y_sampled = df_sampled["loan_status"]

# id is unnecesary to train model and is not a correct parameter
X = X.drop(columns=['id'])

# Convert all parameters to double to be able to train the models
X = X.astype('double[pyarrow]')

# Dataset Split
X_train = X_sampled
y_train = y_sampled

X_eval = X.loc[eval_indexes]
y_eval = y.loc[eval_indexes]

X_test = X.loc[test_indexes]
y_test = y.loc[test_indexes]

# Metrics Evaluation
def evaluate_model(model_name, y_true, y_pred):
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='weighted'),
        'F1 Score': f1_score(y_true, y_pred, average='weighted'),
        'MCC': matthews_corrcoef(y_true, y_pred)
    }
    return metrics

Train Ensemble Models:

In [21]:
# Ensemble 1: Random Forest
def objective_rf(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 15),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1
    }

    # Initialize and train the model
    model = RandomForestClassifier(**param)
    model.fit(X_train, y_train)

    # Predict on the evaluation set (NOT the test set)
    preds = model.predict(X_eval)
    
    # We optimize to maximize MCC (Best for multiclass imbalance)
    mcc = matthews_corrcoef(y_eval, preds)
    return mcc

# --- OPTUNA EXECUTION ---
print("Starting Optuna study for Random Forest...")

# Create study and run optimization
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=3) # We can change the n of trials if we want

# Example of final evaluation:
best_rf = RandomForestClassifier(**study_rf.best_params, class_weight='balanced', random_state=42, n_jobs=-1)
best_rf.fit(X_train, y_train)
rf_preds = best_rf.predict(X_eval)

rf_metrics = evaluate_model("Random Forest", y_eval, rf_preds)
print(rf_metrics)

[I 2026-03-23 13:15:02,599] A new study created in memory with name: no-name-12b758c4-1a31-4917-b86e-642f5e197fe4


Starting Optuna study for Random Forest...


[I 2026-03-23 13:15:06,217] Trial 0 finished with value: 0.21805837867485822 and parameters: {'n_estimators': 80, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 9}. Best is trial 0 with value: 0.21805837867485822.
[I 2026-03-23 13:15:14,433] Trial 1 finished with value: 0.2286001763186447 and parameters: {'n_estimators': 126, 'max_depth': 29, 'min_samples_split': 5, 'min_samples_leaf': 2}. Best is trial 1 with value: 0.2286001763186447.
[I 2026-03-23 13:15:27,252] Trial 2 finished with value: 0.23353687759288524 and parameters: {'n_estimators': 245, 'max_depth': 20, 'min_samples_split': 10, 'min_samples_leaf': 6}. Best is trial 2 with value: 0.23353687759288524.


{'Model': 'Random Forest', 'Accuracy': 0.6387238458039572, 'Precision': 0.7480182028233759, 'Recall': 0.6387238458039572, 'F1 Score': 0.6816615725129805, 'MCC': 0.23353687759288524}


In [22]:
# Ensemble 2: XGBoost
def objective_xgb(trial):
    param = {
        'objective': 'multi:softmax',
        'num_class': 5,               
        'eval_metric': 'mlogloss',    
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': -1
    }

    # 1. Setup Pruning Callback monitoring the eval set
    pruning_callback = XGBoostPruningCallback(trial, 'validation_0-mlogloss')
    
    # 2. Initialize model passing callbacks (and early_stopping) HERE, not in .fit()
    model = xgb.XGBClassifier(
        **param, 
        early_stopping_rounds=10, 
        callbacks=[pruning_callback]
    )

    # 3. Train model 
    model.fit(
        X_train, y_train,
        eval_set=[(X_eval, y_eval)],
        verbose=False
    )

    # Predict on evaluation set
    preds = model.predict(X_eval)
    
    # Maximize MCC
    mcc = matthews_corrcoef(y_eval, preds)
    return mcc

# --- OPTUNA EXECUTION ---
print("Starting Optuna study for XGBoost with Pruner...")

# Example of final evaluation:
study_xgb = optuna.create_study(
    direction='maximize', 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study_xgb.optimize(objective_xgb, n_trials=3) # We can change the n of trials if we want

# Get best parameters and add required static parameters back
best_xgb_params = study_xgb.best_params
best_xgb_params['objective'] = 'multi:softmax'
best_xgb_params['num_class'] = 5
best_xgb_params['random_state'] = 42
best_xgb_params['n_jobs'] = -1

best_xgb = xgb.XGBClassifier(**best_xgb_params)
best_xgb.fit(X_train, y_train)
xgb_preds = best_xgb.predict(X_eval)

# Use the evaluation function from Iteration 1
xgb_metrics = evaluate_model("XGBoost", y_eval, xgb_preds)
print(xgb_metrics)

[I 2026-03-23 13:15:40,525] A new study created in memory with name: no-name-937a69ca-c8a8-4970-a8be-d12ee1fdd88a


Starting Optuna study for XGBoost with Pruner...


[I 2026-03-23 13:16:03,747] Trial 0 finished with value: 0.23892003740336912 and parameters: {'n_estimators': 246, 'max_depth': 9, 'learning_rate': 0.020587396498066783, 'subsample': 0.8275250118218076, 'colsample_bytree': 0.9350276319283991}. Best is trial 0 with value: 0.23892003740336912.
[I 2026-03-23 13:16:07,782] Trial 1 finished with value: 0.11777268784994772 and parameters: {'n_estimators': 101, 'max_depth': 4, 'learning_rate': 0.014985254575026401, 'subsample': 0.7654441220809914, 'colsample_bytree': 0.7913480631802065}. Best is trial 0 with value: 0.23892003740336912.
[I 2026-03-23 13:16:11,507] Trial 2 finished with value: 0.1554696862671437 and parameters: {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.021434644721107957, 'subsample': 0.7500140431055942, 'colsample_bytree': 0.6687727736504846}. Best is trial 0 with value: 0.23892003740336912.


{'Model': 'XGBoost', 'Accuracy': 0.7590544689561064, 'Precision': 0.7356280986691354, 'Recall': 0.7590544689561064, 'F1 Score': 0.7412066480050605, 'MCC': 0.23892003740336912}


In [23]:
# Ensemble 3: Bagging Classifier
def objective_bagging_fast(trial):
    # 1. Base estimator: Decision Tree
    tree_depth = trial.suggest_int('max_depth', 3, 7)
    base_tree = DecisionTreeClassifier(max_depth=tree_depth, random_state=42)

    # 2. Bagging params optimized for low RAM and high speed
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 20, 50),
        'max_samples': trial.suggest_float('max_samples', 0.1, 0.3), 
        'max_features': trial.suggest_float('max_features', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': -1
    }

    # Initialize Bagging Classifier (use 'estimator', or 'base_estimator' for older sklearn)
    model = BaggingClassifier(estimator=base_tree, **param)
    
    # Train model
    model.fit(X_train, y_train)

    # Predict on validation set
    preds = model.predict(X_eval)
    
    # Maximize MCC
    mcc = matthews_corrcoef(y_eval, preds)
    return mcc

# --- OPTUNA EXECUTION ---
print("Starting lightweight Optuna study for Bagging Classifier...")

study_bagging = optuna.create_study(direction='maximize')
# Only 10 trials to keep it short and sweet
study_bagging.optimize(objective_bagging_fast, n_trials=3) # We can change the n of trials if we want

# Example of final evaluation extraction:
# Extract best parameters but isolate max_depth for the base estimator
best_params = study_bagging.best_params
best_depth = best_params.pop('max_depth')

final_base_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
best_bagging = BaggingClassifier(
    estimator=final_base_tree, 
    **best_params, 
    random_state=42, 
    n_jobs=2
)

best_bagging.fit(X_train, y_train)
bagging_preds = best_bagging.predict(X_eval)

bagging_metrics = evaluate_model("Bagging Classifier", y_eval, bagging_preds)
print(bagging_metrics)

[I 2026-03-23 13:16:33,543] A new study created in memory with name: no-name-07a15461-e41c-47a6-83fb-40b40f9b9b6c


Starting lightweight Optuna study for Bagging Classifier...


[I 2026-03-23 13:16:36,806] Trial 0 finished with value: 0.16534778911763137 and parameters: {'max_depth': 5, 'n_estimators': 21, 'max_samples': 0.1544614051116831, 'max_features': 0.800453456911938}. Best is trial 0 with value: 0.16534778911763137.
[I 2026-03-23 13:16:40,623] Trial 1 finished with value: 0.19006291139369563 and parameters: {'max_depth': 7, 'n_estimators': 45, 'max_samples': 0.2343831256223256, 'max_features': 0.7690222772022488}. Best is trial 1 with value: 0.19006291139369563.
[I 2026-03-23 13:16:43,854] Trial 2 finished with value: 0.19310695236592149 and parameters: {'max_depth': 7, 'n_estimators': 20, 'max_samples': 0.20734837215394258, 'max_features': 0.9512208679765687}. Best is trial 2 with value: 0.19310695236592149.


{'Model': 'Bagging Classifier', 'Accuracy': 0.7477683647941779, 'Precision': 0.7231162720838479, 'Recall': 0.7477683647941779, 'F1 Score': 0.7253868414941457, 'MCC': 0.19310695236592149}


In [ ]:
# Ensemble 4: Voting Classifier

print("Extracting base predictions once to save time...")

# 1. Get predictions directly from your PRE-FITTED models
preds_rf = best_rf.predict(X_eval)
preds_xgb = best_xgb.predict(X_eval)
preds_bag = best_bagging.predict(X_eval)

# Stack them together into a matrix of shape (n_samples, 3)
all_preds = np.column_stack((preds_rf, preds_xgb, preds_bag)).astype(int)

def objective_voting_smart(trial):
    # Suggest weights
    w_rf = trial.suggest_int('w_rf', 1, 5)
    w_xgb = trial.suggest_int('w_xgb', 1, 5)
    w_bag = trial.suggest_int('w_bagging', 1, 5)
    weights = [w_rf, w_xgb, w_bag]
    
    # Manual and lightning-fast "Hard Voting" simulation
    final_preds = []
    for row in all_preds:
        # np.bincount sums the weights for each class and argmax picks the winner
        winner = np.bincount(row, weights=weights).argmax()
        final_preds.append(winner)
        
    # Maximize MCC
    mcc = matthews_corrcoef(y_eval, final_preds)
    return mcc

# --- OPTUNA EXECUTION ---
print("Starting ultra-fast Optuna study for Voting weights...")

study_voting = optuna.create_study(direction='maximize')
study_voting.optimize(objective_voting_smart, n_trials=30) # We can change the n of trials if we want

# Extract best weights and train final Voting model on once

best_voting_params = study_voting.best_params
final_weights = [
    best_voting_params['w_rf'], 
    best_voting_params['w_xgb'], 
    best_voting_params['w_bagging']
]

print(f"\nBest weights found -> RF: {final_weights[0]}, XGB: {final_weights[1]}, Bagging: {final_weights[2]}")

# Now we create the Scikit-Learn object and fit it JUST ONCE
best_voting = VotingClassifier(
    estimators=[
        ('RandomForest', best_rf),
        ('XGBoost', best_xgb),
        ('FastBagging', best_bagging)
    ],
    voting='hard',
    weights=final_weights,
    n_jobs=1
)

print("Training final Voting Classifier")
best_voting.fit(X_train, y_train)

# Predict and evaluate
voting_preds = best_voting.predict(X_eval)
voting_metrics = evaluate_model("Voting Classifier", y_eval, voting_preds)

print("\nFinal Voting Classifier Metrics:")
print(voting_metrics)

Extracting base predictions once to save time...


[I 2026-03-23 13:18:05,084] A new study created in memory with name: no-name-0fb762c7-1133-4445-a770-3babd19ca493
[I 2026-03-23 13:18:05,212] Trial 0 finished with value: 0.23693636415456584 and parameters: {'w_rf': 2, 'w_xgb': 5, 'w_bagging': 3}. Best is trial 0 with value: 0.23693636415456584.


Starting ultra-fast Optuna study for Voting weights...


[I 2026-03-23 13:18:05,340] Trial 1 finished with value: 0.19310695236592149 and parameters: {'w_rf': 1, 'w_xgb': 1, 'w_bagging': 5}. Best is trial 0 with value: 0.23693636415456584.
[I 2026-03-23 13:18:05,463] Trial 2 finished with value: 0.22218205695349305 and parameters: {'w_rf': 2, 'w_xgb': 4, 'w_bagging': 5}. Best is trial 0 with value: 0.23693636415456584.
[I 2026-03-23 13:18:05,589] Trial 3 finished with value: 0.23892003740336912 and parameters: {'w_rf': 2, 'w_xgb': 5, 'w_bagging': 1}. Best is trial 3 with value: 0.23892003740336912.
[I 2026-03-23 13:18:05,713] Trial 4 finished with value: 0.19365352216058354 and parameters: {'w_rf': 2, 'w_xgb': 1, 'w_bagging': 3}. Best is trial 3 with value: 0.23892003740336912.
[I 2026-03-23 13:18:05,837] Trial 5 finished with value: 0.23892003740336912 and parameters: {'w_rf': 2, 'w_xgb': 5, 'w_bagging': 1}. Best is trial 3 with value: 0.23892003740336912.
[I 2026-03-23 13:18:05,962] Trial 6 finished with value: 0.2313174088247843 and param


Best weights found -> RF: 2, XGB: 5, Bagging: 1
Training final Voting Classifier (Only doing this ONCE!)...

Final Voting Classifier Metrics:
{'Model': 'Voting Classifier', 'Accuracy': 0.7590544689561064, 'Precision': 0.7356280986691354, 'Recall': 0.7590544689561064, 'F1 Score': 0.7412066480050605, 'MCC': 0.23892003740336912}


Model Comparison:

In [26]:
# Final Comparison Table

# We gather the dictionaries generated by our evaluate_model function
# and turn them into a beautiful Pandas DataFrame.

# Make sure these variables match whatever you named them in previous steps!
all_metrics = [
    rf_metrics,       # From Iteration 1
    xgb_metrics,      # From Iteration 2
    bagging_metrics,  # From Iteration 3
    voting_metrics    # From Iteration 4
]

comparison_df = pd.DataFrame(all_metrics)

print("\n" + "="*70)
print("FINAL MODEL COMPARISON (SORTED BY MCC)")
print("="*70)
# Sort by MCC descending to see the winner at the top
comparison_df_sorted = comparison_df.sort_values(by='MCC', ascending=False)
print(comparison_df_sorted.to_string(index=False))


FINAL MODEL COMPARISON (SORTED BY MCC)
             Model  Accuracy  Precision   Recall  F1 Score      MCC
           XGBoost  0.759054   0.735628 0.759054  0.741207 0.238920
 Voting Classifier  0.759054   0.735628 0.759054  0.741207 0.238920
     Random Forest  0.638724   0.748018 0.638724  0.681662 0.233537
Bagging Classifier  0.747768   0.723116 0.747768  0.725387 0.193107
